In [1]:
#Load the Base Dataset
import pandas as pd

df = pd.read_csv("../data/processed/base_transactions.csv")
df["TransactionDT"] = pd.to_datetime(df["TransactionDT"])

df = df.sort_values(["customer_id", "TransactionDT"])
df.head()


,TransactionDT,TransactionAmt,customer_id,card1,card2,ProductCD,isFraud
180332,1970-02-16 21:14:11,29.000,10000_111.0,10000,111.0,W,0
335565,1970-04-08 11:23:35,39.394,10003_555.0,10003,555.0,C,0
344341,1970-04-10 22:34:42,10.755,10003_555.0,10003,555.0,C,0
344362,1970-04-10 22:40:15,19.093,10003_555.0,10003,555.0,C,0
344815,1970-04-11 00:40:05,19.093,10003_555.0,10003,555.0,C,0


In [2]:
#Set Index (Required for Rolling Windows)
df = df.set_index("TransactionDT")

In [3]:
#Transaction Count (Velocity Features)
#Transactions in last 1 hour
df["txn_count_1h"] = (
    df.groupby("customer_id")
      .rolling("1h")["TransactionAmt"]
      .count()
      .reset_index(level=0, drop=True)
)


In [4]:
#Transactions in last 24 hours
df["txn_count_24h"] = (
    df.groupby("customer_id")
      .rolling("24h")["TransactionAmt"]
      .count()
      .reset_index(level=0, drop=True)
)

In [5]:
#Transactions in last 7 days
df["txn_count_7d"] = (
    df.groupby("customer_id")
      .rolling("7D")["TransactionAmt"]
      .count()
      .reset_index(level=0, drop=True)
)

In [6]:
#Amount-Based Features (Spending Behavior)
#Average amount (1h / 24h / 7d)
df["avg_amt_1h"] = (
    df.groupby("customer_id")
      .rolling("1h")["TransactionAmt"]
      .mean()
      .reset_index(level=0, drop=True)
)

df["avg_amt_24h"] = (
    df.groupby("customer_id")
      .rolling("24h")["TransactionAmt"]
      .mean()
      .reset_index(level=0, drop=True)
)

df["avg_amt_7d"] = (
    df.groupby("customer_id")
      .rolling("7D")["TransactionAmt"]
      .mean()
      .reset_index(level=0, drop=True)
)


In [7]:
#Max amount (fraud spikes!)
df["max_amt_24h"] = (
    df.groupby("customer_id")
      .rolling("24h")["TransactionAmt"]
      .max()
      .reset_index(level=0, drop=True)
)

In [8]:
#Amount Deviation - Comparing current transaction vs customer baseline
df["amount_dev_24h"] = (
    df["TransactionAmt"] - df["avg_amt_24h"]
)


In [9]:
#Z-Score (Normalized deviation)
df["std_amt_24h"] = (
    df.groupby("customer_id")
      .rolling("24h")["TransactionAmt"]
      .std()
      .reset_index(level=0, drop=True)
)

df["amount_zscore_24h"] = (
    df["amount_dev_24h"] / df["std_amt_24h"]
)


In [10]:
#Time-of-Day Risk Features
df["hour"] = df.index.hour
df["is_night_txn"] = df["hour"].isin([0,1,2,3,4]).astype(int)

In [11]:
#Clean Up NaNs
df_feat = df.dropna().reset_index()
df_feat.head()

,TransactionDT,TransactionAmt,customer_id,card1,card2,ProductCD,isFraud,txn_count_1h,txn_count_24h,txn_count_7d,avg_amt_1h,avg_amt_24h,avg_amt_7d,max_amt_24h,amount_dev_24h,std_amt_24h,amount_zscore_24h,hour,is_night_txn
0,1970-04-10 22:40:15,19.093,10003_555.0,10003,555.0,C,0,2.0,2.0,3.0,14.924,14.924000,23.080667,19.093,4.169000,5.895856,0.707107,22,0
1,1970-04-11 00:40:05,19.093,10003_555.0,10003,555.0,C,0,1.0,3.0,4.0,19.093,16.313667,22.083750,19.093,2.779333,4.813947,0.577350,0,1
2,1970-01-16 18:25:07,100.000,10004_529.0,10004,529.0,R,0,1.0,2.0,2.0,100.000,125.000000,125.000000,150.000,-25.000000,35.355339,-0.707107,18,0
3,1970-02-13 19:58:39,107.950,10004_529.0,10004,529.0,W,0,1.0,2.0,3.0,107.950,316.450000,736.133333,524.950,-208.500000,294.863528,-0.707107,19,0
4,1970-02-16 09:20:45,54.490,10004_529.0,10004,529.0,W,0,1.0,2.0,4.0,54.490,42.745000,179.597500,54.490,11.745000,16.609938,0.707107,9,0


In [12]:
#Quick Sanity Check
df_feat[
    ["txn_count_1h", "txn_count_24h", "avg_amt_24h", "amount_zscore_24h"]
].describe()


,txn_count_1h,txn_count_24h,avg_amt_24h,amount_zscore_24h
count,441762.000000,441762.000000,441762.000000,441762.000000
mean,2.892297,25.773548,133.266799,-0.003203
std,6.285429,51.232329,126.537487,0.939224
min,1.000000,2.000000,2.300000,-6.283292
25%,1.000000,4.000000,70.330455,-0.656172
50%,2.000000,11.000000,104.475000,-0.315966
75%,3.000000,27.000000,153.099583,0.577350
max,192.000000,881.000000,4906.320000,9.776528


In [13]:
#Save Feature Dataset
df_feat.to_csv(
    "../data/processed/time_window_features.csv",
    index=False
)